In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_testing_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "precip_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"


matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2904_123501', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_222_2904_124105', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_333_2904_124709', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_444_2904_125314', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_555_2904_125917', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_666_2904_130523', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_777_2904_131125', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_888_2904_131729']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['QObs_mm_d_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy(deep=True)
    xr_ensemble['QObs_mm_d_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'CAMELS_UY_10': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:        (date: 3652, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 15kB 6.151 5.395 ... 0.2979 0.2661
       QObs_mm_d_sim  (date, time_step) float32 15kB 0.9138 0.7933 ... 1.054 0.8844}},
 'CAMELS_UY_11': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:        (date: 3652, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 15kB 0.4463 0.8222 ... 0.6856
       QObs_mm_d_sim  (date, time_step) float32 15kB 0.7428 1.319 ... 0.5296 0.4919}},
 'CAMELS_UY_15': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:        (date: 3652, time_step: 1)
   Coordinates:
     * date    

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}

for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['QObs_mm_d_obs']
    sim = xr_ds['QObs_mm_d_sim']
    
    # Skip basin if all obs or sim are NaN
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = calculate_metrics(
        obs=obs,
        sim=sim,
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

Skipping CAMELS_UY_15 — all observed/simulated values are NaN


,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
CAMELS_UY_10,0.754601,0.657869,0.811091,0.878102,1.009018,0.878437,0.999450,-0.000403,7.221441,-3.373981,-260.898865,1.461538,0.540541,31.909439
CAMELS_UY_11,0.590598,3.564737,1.888051,0.610236,0.724743,0.774639,0.840745,-0.083528,-29.408249,36.798660,-65.631454,0.625000,0.450980,53.599358
CAMELS_UY_16,0.718075,4.205794,2.050803,0.813136,0.891067,0.848468,1.009485,0.002844,-7.621765,-8.253308,-801.675964,0.350000,0.400000,38.444866
CAMELS_UY_2,0.414846,3.915557,1.978777,0.448875,1.346728,0.852064,1.402037,0.257825,38.794609,-0.953025,-828.869446,1.444444,0.731707,38.033237
CAMELS_UY_3,0.723366,2.372184,1.540190,0.726657,1.131906,0.891625,1.213476,0.117874,14.687284,-14.114785,-38.542835,0.750000,0.446809,31.232327
CAMELS_UY_5,0.627542,7.013366,2.648276,0.670667,1.140871,0.851266,1.257864,0.115063,14.707029,-11.086125,-742.868713,0.705882,0.565217,35.489662
CAMELS_UY_6,0.802618,1.430566,1.196063,0.755629,1.115193,0.924443,1.201838,0.124861,13.554676,-9.020659,5.689001,1.052632,0.636364,23.500290
CAMELS_UY_7,0.697952,5.237692,2.288601,0.809778,1.104404,0.869116,1.090296,0.045274,8.400922,-10.296971,-239.891785,0.850000,0.425532,39.131184
CAMELS_UY_8,0.626925,5.053152,2.247922,0.776861,1.105102,0.837274,1.110748,0.048692,12.173225,-15.265359,-121.593056,0.947368,0.392157,37.364651


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(ensemble_metrics_dir/f"{save_name}.csv")

In [7]:
df_metrics.median()

NSE               0.708014
MSE               3.740147
RMSE              1.933414
KGE               0.766245
Alpha-NSE         1.104753
Pearson-r         0.860590
Beta-KGE          1.100522
Beta-NSE          0.046983
FHV              10.287074
FMS              -9.658815
FLV            -180.742420
Peak-Timing       0.898684
Missed-Peaks      0.495760
Peak-MAPE        36.427156
dtype: float64